In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [2]:
train_df = pd.read_csv("/content/GSE98320.csv")
val_df   = pd.read_csv("/content/GSE129166.csv")

X_train = train_df.drop(columns=["sample_id", "diagnosis"])
y_train = train_df["diagnosis"]
X_val   = val_df.drop(columns=["sample_id", "diagnosis"])[X_train.columns]
y_val   = val_df["diagnosis"]

In [3]:
class_labels = sorted(y_train.unique())

clf = RandomForestClassifier(random_state=42)
param_grid = {
    "n_estimators": [200, 500],
    "max_depth": [None, 10, 20],
    "max_features": ["sqrt", "log2"],
}

In [4]:
def print_metrics(y_true, y_pred, label):
    print(f"\n=== RANDOM FOREST — {label} ===")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=class_labels, zero_division=0)
    for cls, pi, ri, fi in zip(class_labels, p, r, f):
        print(f"  {cls:6s} | precision={pi:.4f}  recall={ri:.4f}  f1={fi:.4f}")
    print(f"  MACRO  | precision={np.mean(p):.4f}  recall={np.mean(r):.4f}  f1={np.mean(f):.4f}")

outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
X_train_r, y_train_r = X_train.reset_index(drop=True), y_train.reset_index(drop=True)
oof_pred = np.empty(len(y_train_r), dtype=object)

In [5]:
for fold, (tr_idx, te_idx) in enumerate(outer_cv.split(X_train_r, y_train_r)):
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    search = GridSearchCV(clone(clf), param_grid, cv=inner_cv, scoring="accuracy", n_jobs=-1)
    search.fit(X_train_r.iloc[tr_idx], y_train_r.iloc[tr_idx])
    oof_pred[te_idx] = search.predict(X_train_r.iloc[te_idx])
    print(f"  fold {fold+1}/10 done, best params={search.best_params_}")

print_metrics(y_train_r, oof_pred, "Cross-Validation Performance (GSE98320)")

  fold 1/10 done, best params={'max_depth': 10, 'max_features': 'sqrt', 'n_estimators': 200}
  fold 2/10 done, best params={'max_depth': None, 'max_features': 'sqrt', 'n_estimators': 200}
  fold 3/10 done, best params={'max_depth': None, 'max_features': 'sqrt', 'n_estimators': 200}
  fold 4/10 done, best params={'max_depth': 10, 'max_features': 'sqrt', 'n_estimators': 200}
  fold 5/10 done, best params={'max_depth': None, 'max_features': 'sqrt', 'n_estimators': 200}
  fold 6/10 done, best params={'max_depth': None, 'max_features': 'sqrt', 'n_estimators': 500}
  fold 7/10 done, best params={'max_depth': None, 'max_features': 'sqrt', 'n_estimators': 200}
  fold 8/10 done, best params={'max_depth': None, 'max_features': 'sqrt', 'n_estimators': 200}
  fold 9/10 done, best params={'max_depth': 10, 'max_features': 'sqrt', 'n_estimators': 200}
  fold 10/10 done, best params={'max_depth': 10, 'max_features': 'sqrt', 'n_estimators': 200}

=== RANDOM FOREST — Cross-Validation Performance (GSE983

In [6]:
final_search = GridSearchCV(clone(clf), param_grid,
                             cv=StratifiedKFold(3, shuffle=True, random_state=42),
                             scoring="accuracy", n_jobs=-1)
final_search.fit(X_train, y_train)
val_pred = final_search.predict(X_val)
print_metrics(y_val, val_pred, "Independent Validation Performance (GSE129166)")


=== RANDOM FOREST — Independent Validation Performance (GSE129166) ===
Accuracy: 0.9351
  ABMR   | precision=0.8125  recall=0.8667  f1=0.8387
  NR     | precision=0.9828  recall=0.9500  f1=0.9661
  TCMR   | precision=0.6667  recall=1.0000  f1=0.8000
  MACRO  | precision=0.8206  recall=0.9389  f1=0.8683
